In [ ]:
MODEL = 'hydra'
EPOCHS = 500
BATCH_SIZE = 8

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--index-url', 'https://download.pytorch.org/whl/cu121',
                       'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('rl_switch_windows_train.npz'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WIN_TRAIN = AUX_DIR / 'rl_switch_windows_train.npz'
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
print('Setup ready')

In [ ]:
OUT_DIR = WORK / 'runs' / MODEL
OUT_DIR.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_hydra_pro.py',
        '--windows_train', str(WIN_TRAIN),
        '--out_dir', str(OUT_DIR),
        '--madt_epochs', str(EPOCHS),
        '--iql_epochs', str(EPOCHS),
        '--mappo_steps', str(EPOCHS * 10),
        '--batch', str(BATCH_SIZE),
        '--device', 'cuda']
print('>>>', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'training failed'

In [ ]:
for root, dirs, files in os.walk(WORK):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.pth', '.json', '.csv']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')